# Pipeline Stage 1 — Ground Filtering (v9, batch timestamps)

Same job as v7/v8: run Patchwork++ on a LiDAR frame and save the surviving
ground points as a `.bin` that Notebook 2 can consume. What changed in v9:

1. **Batch of timestamps in one run.** Instead of editing a single
   `MCAP_FRAME_TIMESTAMP` and re-running everything, put a *list* in
   `MCAP_FRAME_TIMESTAMPS`. The bag is indexed once (cached) and every
   timestamp is loaded, segmented and exported inside one loop — no
   re-loading the whole bag per frame.
2. **Three outputs per frame**, not two:
   * `*_raw.bin`     — the scene **BEFORE** Patchwork++ (calibrated into
     `base_link` for MCAP). This is the unfiltered reference cloud.
   * `*_pw.bin`      — ground-only, **AFTER** Patchwork++ (Notebook 2 input).
   * `*_pw_full.bin` — full scene after Patchwork++, class in intensity
     (`0.0` = ground, `1.0` = obstacle). Foxglove / inspection only.
   Drop the `_raw` and `_pw_full` clouds into Foxglove together to *see*
   exactly what the filter kept vs threw away.

| Filter notebook produces                | Calculation notebook uses |
|---|---|
| Patchwork++ only                        | PCA · RANSAC · height-derivative |
| Patchwork++ + GLE post-filter           | PCA · RANSAC · height-derivative |
| RANSAC pipeline (collaborator's)        | PCA · RANSAC · height-derivative |

### Analogy — the cartographer's contact sheet

v7 handed the cartographer only the *traced roads* (ground). v8 added the
full *aerial photo* with every pixel tagged. v9 is the whole **contact
sheet**: for each moment in time you get the untouched photo (`_raw`), the
traced roads (`_pw`), and the tagged photo (`_pw_full`) — all developed in
one pass instead of one negative at a time.

> **Warning.** The GLE post-filter rejects high-curvature patches — exactly
> the signal we're trying to *measure* downstream. Default thresholds are
> intentionally lenient. Defend whatever values you publish.

In [ ]:
# Run once if you don't have these:
# !pip install numpy matplotlib pypatchworkpp
# !pip install rosbags          # only needed for MCAP support

import os
import numpy as np
import matplotlib.pyplot as plt
import pypatchworkpp

np.set_printoptions(suppress=True, precision=4)


# ==========================================================
# WORKFLOW SELECTION
# ----------------------------------------------------------
#   "bin"   -> nuScenes-format .bin       (N, 5)  [x,y,z,intensity,ring]
#   "mcap"  -> ROS bag, multi-LiDAR       PointCloud2 messages
#   "kitti" -> KITTI Velodyne .bin        (N, 4)  [x,y,z,intensity]
# ==========================================================
WORKFLOW = "mcap"          # "bin", "mcap", or "kitti"


# ==========================================================
# OUTPUT paths + what to write per frame
# ----------------------------------------------------------
#   EXPORT_RAW_CLOUD   -> *_raw.bin      BEFORE Patchwork++ (reference)
#   EXPORT_GROUND      -> *_pw.bin       ground only        (Notebook 2 input)
#   EXPORT_FULL_CLOUD  -> *_pw_full.bin  ground+obstacle    (labelled, Foxglove)
# ==========================================================
EXPORT_RAW_CLOUD   = True    # NEW: unfiltered input cloud, for before/after
EXPORT_GROUND      = True
EXPORT_FULL_CLOUD  = False
PLOT_EACH_FRAME    = True    # before/after plots for every timestamp
GROUND_BIN_DIR     = r"C:\Users\karel\OneDrive\BEP"
GROUND_BIN_BASE    = "ground_points"


# ==========================================================
# OPTIONAL POST-FILTER (Patchwork++ GLE-style)
# ----------------------------------------------------------
# After Patchwork++ runs, lay a coarse grid over the ground
# points, fit a PCA plane in each cell, and reject the whole
# cell if either:
#   - flatness  sigma = lambda3 / (lambda1+lambda2+lambda3)
#                exceeds FLATNESS_THRESHOLD     (cell too "fluffy")
#   - uprightness |n_z|
#                falls below UPRIGHTNESS_THRESHOLD (normal too tilted)
#
# WARNING: on a curvature-focused pipeline this can throw out
# the very signal you want. Keep FLATNESS_THRESHOLD >= 0.02
# unless you have a good reason.
# ==========================================================
APPLY_EIGEN_FLATNESS_FILTER = False

POST_CELL_SIZE          = 1.0     # [m]
POST_MIN_PTS_PER_CELL   = 12
FLATNESS_THRESHOLD      = 0.03
UPRIGHTNESS_THRESHOLD   = 0.85    # ~31 deg tilt


# ==========================================================
# Frame selection (MCAP only) -- BATCH of timestamps
# ----------------------------------------------------------
# Put as many timestamps (Unix seconds) as you like in the list.
# The bag is indexed ONCE and every timestamp is processed in the
# same run, so you never re-load the whole bag per timestamp.
#
# For "bin"/"kitti" this list is ignored (one file = one frame).
# ==========================================================
MCAP_FRAME_TIMESTAMPS = [
    1779440435.640716709,
]
MCAP_FRAME_INDICES = None     # optional: [12, 57, 130] instead of timestamps
MCAP_TOPIC         = None      # None = use all LiDAR topics + calibrate


# ==========================================================
# Sensor configuration -- single source of truth
# ----------------------------------------------------------
# sensor_height is the LiDAR origin's height above the ground.
# Patchwork++ uses it as the prior for where ground points live.
# ==========================================================
_SENSOR_HEIGHT_BY_WORKFLOW = {
    "bin":   1.84,   # nuScenes LIDAR_TOP
    "mcap":  0.90,   # SenseBike
    "kitti": 1.73,   # KITTI Velodyne HDL-64E on VW Passat roof
}
SENSOR_CONFIG = {
    "sensor_height": _SENSOR_HEIGHT_BY_WORKFLOW[WORKFLOW],
    "min_range":   1.0,
    "max_range":  50.0,
    "verbose":     False,
}


# ==========================================================
# I/O paths
# ==========================================================
BIN_PATH   = r"C:\Users\JRepa\OneDrive - Delft University of Technology\Documenten\02. TU Delft\2025-2026\BEP\Data\Pointclouds curvature\pointcloud.bin"
MCAP_PATH  = r"C:\Users\karel\OneDrive\BEP\rosbag_0.mcap"
KITTI_PATH = r"D:\ONLY non-flat surfaces\Pointclouds\kuitbrug_2026_05_22_10_50_00_map.bin"

LIDAR_TOPICS = [
    "/rslidar/M1P_deskewed",
    "/rslidar/helios_R",
    "/rslidar/helios_L",
]

BASE_FRAME = "base_link"

## 1. Loaders

Three formats are supported:

* **nuScenes `.bin`** — `(N, 5)` float32 → `[x, y, z, intensity, ring]`.
* **KITTI `.bin`**   — `(N, 4)` float32 → `[x, y, z, intensity]`. Already in the
  Velodyne frame (x-forward, y-left, z-up), so no rotation is needed.
* **`.mcap`**        — ROS bag with one or more `sensor_msgs/PointCloud2`
  topics. Each topic is run through Patchwork++ *in its own sensor frame*
  first; `/tf_static` is applied *afterwards* to move the results into
  `base_link`. (Applying it first would shift z-values and break Patchwork++'s
  `sensor_height` prior.)

All paths return an `(N, 3)` `float64` XYZ array.

In [ ]:
# ------------------------------------------------------------
# Loader 1: nuScenes-format .bin file
# ------------------------------------------------------------
def load_nuscenes_bin(path):
    """Load (N, 3) XYZ from a nuScenes-format .bin file."""
    pts = np.fromfile(path, dtype=np.float32).reshape(-1, 5)
    return pts[:, :3].astype(np.float64)


# ------------------------------------------------------------
# Loader 2: KITTI Velodyne .bin file (raw / odometry / SemanticKITTI)
# ------------------------------------------------------------
def load_kitti_bin(path):
    """Load (N, 3) XYZ from a KITTI-format .bin file.
    KITTI layout: float32, (N, 4) = [x, y, z, reflectance].
    Coordinate frame is already x-forward, y-left, z-up (Velodyne).
    """
    pts = np.fromfile(path, dtype=np.float32).reshape(-1, 4)
    return pts[:, :3].astype(np.float64)


# ------------------------------------------------------------
# Loader 3: a single frame from a ROS .mcap bag
# ------------------------------------------------------------
_PF_DTYPE = {1: ("i1", 1), 2: ("u1", 1), 3: ("i2", 2), 4: ("u2", 2),
             5: ("i4", 4), 6: ("u4", 4), 7: ("f4", 4), 8: ("f8", 8)}


def _pointcloud2_to_xyz(msg):
    """Extract (xyz, frame_id) from a PointCloud2 message."""
    field_map = {f.name: f for f in msg.fields}
    missing = [k for k in ("x", "y", "z") if k not in field_map]
    if missing:
        raise RuntimeError(f"PointCloud2 missing {missing}. Have: {list(field_map)}")

    n = msg.width * msg.height
    raw = np.frombuffer(msg.data, dtype=np.uint8).reshape(n, msg.point_step)

    xyz = np.empty((n, 3), dtype=np.float64)
    for i, name in enumerate(("x", "y", "z")):
        f = field_map[name]
        dtype_str, size = _PF_DTYPE[f.datatype]
        col_bytes = raw[:, f.offset:f.offset + size].copy()
        xyz[:, i] = col_bytes.view(np.dtype(dtype_str)).flatten().astype(np.float64)

    valid = np.isfinite(xyz).all(axis=1)
    return xyz[valid], msg.header.frame_id


# ============================================================
# EXTRINSIC CALIBRATION helpers (/tf_static) -- MCAP only
# ------------------------------------------------------------
# Applied AFTER Patchwork++ so the sensor_height prior is not
# broken by shifting z values.
# ============================================================
_TF_TREE_CACHE = {}


def _quat_to_R(x, y, z, w):
    n = x*x + y*y + z*z + w*w
    if n < 1e-12:
        return np.eye(3)
    s = 2.0 / n
    return np.array([
        [1 - s*(y*y + z*z),     s*(x*y - z*w),     s*(x*z + y*w)],
        [    s*(x*y + z*w), 1 - s*(x*x + z*z),     s*(y*z - x*w)],
        [    s*(x*z - y*w),     s*(y*z + x*w), 1 - s*(x*x + y*y)],
    ])


def _build_tf_tree(mcap_path):
    from rosbags.highlevel import AnyReader
    from pathlib import Path

    if mcap_path in _TF_TREE_CACHE:
        return _TF_TREE_CACHE[mcap_path]

    tree = {}
    with AnyReader([Path(mcap_path)]) as reader:
        wanted = [c for c in reader.connections if c.topic == "/tf_static"]
        if not wanted:
            print("[tf] WARNING: /tf_static not found -- no calibration available.")
            _TF_TREE_CACHE[mcap_path] = tree
            return tree
        for conn, ts, raw in reader.messages(connections=wanted):
            msg = reader.deserialize(raw, conn.msgtype)
            for tf in msg.transforms:
                parent = tf.header.frame_id
                child  = tf.child_frame_id
                t = np.array([tf.transform.translation.x,
                              tf.transform.translation.y,
                              tf.transform.translation.z])
                q = tf.transform.rotation
                R = _quat_to_R(q.x, q.y, q.z, q.w)
                tree[child] = (parent, R, t)
    print(f"[tf] Parsed {len(tree)} static transforms. Frames: {sorted(tree.keys())}")
    _TF_TREE_CACHE[mcap_path] = tree
    return tree


def _resolve_to_base(tree, child_frame, base_frame="base_link", max_depth=10):
    R_acc = np.eye(3); t_acc = np.zeros(3); cur = child_frame
    for _ in range(max_depth):
        if cur == base_frame:
            return R_acc, t_acc
        if cur not in tree:
            raise RuntimeError(f"'{cur}' not in /tf_static -- cannot reach '{base_frame}'.")
        parent, R, t = tree[cur]
        t_acc = R @ t_acc + t
        R_acc = R @ R_acc
        cur = parent
    raise RuntimeError(f"TF chain too deep from '{child_frame}'.")


def _apply_tf(xyz, R, t):
    return xyz @ R.T + t


# ============================================================
# Fast MCAP loader -- caches per-topic clouds for post-PW++
# calibration in _LAST_LOAD_CACHE.
# ============================================================
_MCAP_INDEX_CACHE = {}
_LAST_LOAD_CACHE  = {}


def _build_mcap_index(path):
    from rosbags.highlevel import AnyReader
    from pathlib import Path

    if path in _MCAP_INDEX_CACHE:
        return _MCAP_INDEX_CACHE[path]

    print("[mcap] Indexing bag (one-time)... ", end="", flush=True)
    index = {}
    with AnyReader([Path(path)]) as reader:
        pc2_conns = [c for c in reader.connections
                     if c.msgtype == "sensor_msgs/msg/PointCloud2"]
        for c in pc2_conns:
            index[c.topic] = []
        for conn, ts, raw in reader.messages(connections=pc2_conns):
            index[conn.topic].append(ts)
    for topic in index:
        index[topic] = np.asarray(index[topic], dtype=np.int64)
    _MCAP_INDEX_CACHE[path] = {"index": index}
    print(f"done. Topics: {list(index.keys())}")
    return _MCAP_INDEX_CACHE[path]


def _resolve_frame_time(path, target_idx, target_sec, ref_topic):
    cache = _build_mcap_index(path)
    ts_list = cache["index"][ref_topic]
    if target_sec is not None:
        target_ns = int(target_sec * 1e9)
        i = int(np.argmin(np.abs(ts_list - target_ns)))
        diff_s = abs(ts_list[i] - target_ns) / 1e9
        print(f"[mcap] Timestamp {target_sec:.3f} -> frame {i} on {ref_topic} "
              f"(off by {diff_s:.4f} s)")
        return ts_list[i], i
    if target_idx is not None:
        return ts_list[target_idx], target_idx
    raise ValueError("Need MCAP_FRAME_INDEX or MCAP_FRAME_TIMESTAMP.")


def load_mcap_frame_fast(path, topics, target_idx=None, target_sec=None,
                         tol_sec=0.05):
    from rosbags.highlevel import AnyReader
    from pathlib import Path

    ref_topic = topics[0]
    target_ns, ref_idx = _resolve_frame_time(path, target_idx, target_sec, ref_topic)

    tol_ns = int(tol_sec * 1e9)
    start_ns = target_ns - tol_ns
    end_ns   = target_ns + tol_ns

    out = {t: None for t in topics}

    with AnyReader([Path(path)]) as reader:
        wanted = [c for c in reader.connections if c.topic in set(topics)]
        for conn, ts, raw in reader.messages(connections=wanted,
                                             start=start_ns,
                                             stop=end_ns + 1):
            if out[conn.topic] is None:
                msg = reader.deserialize(raw, conn.msgtype)
                xyz, frame_id = _pointcloud2_to_xyz(msg)
                out[conn.topic] = (xyz, frame_id)
                if all(v is not None for v in out.values()):
                    break

    _LAST_LOAD_CACHE.clear()

    parts = []
    for t in topics:
        if out[t] is None:
            print(f"  [warn] {t}: no message within +/-{tol_sec*1000:.0f} ms")
        else:
            xyz, fid = out[t]
            _LAST_LOAD_CACHE[t] = (xyz, fid)
            print(f"  {t}: {len(xyz):,} pts  (frame: {fid})")
            parts.append(xyz)

    if not parts:
        raise RuntimeError("No LiDAR frames matched.")

    merged = np.concatenate(parts, axis=0)
    print(f"  Merged total: {len(merged):,} points (raw, uncalibrated)")
    return merged


def load_mcap_frame(path, topic=None, frame_index=0, list_topics=False):
    from rosbags.highlevel import AnyReader
    from pathlib import Path

    cache = _build_mcap_index(path)
    if list_topics:
        print("PointCloud2 topics in this bag:")
        for t, ts in cache["index"].items():
            print(f"  {t}  ({len(ts)} messages)")
        return None

    if topic is None:
        topic = next(iter(cache["index"].keys()))
        print(f"[mcap] Auto-selected topic: {topic}")

    target_ns = cache["index"][topic][frame_index]
    with AnyReader([Path(path)]) as reader:
        wanted = [c for c in reader.connections if c.topic == topic]
        for conn, ts, raw in reader.messages(connections=wanted,
                                             start=target_ns,
                                             stop=target_ns + 1):
            msg = reader.deserialize(raw, conn.msgtype)
            xyz, fid = _pointcloud2_to_xyz(msg)
            print(f"[mcap] Loaded frame {frame_index} from {topic} (frame: {fid})")
            return xyz
    raise RuntimeError("Frame not found.")


# ============================================================
# Dispatcher -- timestamp-aware (MCAP); bin/kitti ignore the timestamp
# ============================================================
def load_frame(timestamp=None, frame_index=None):
    """Load ONE raw frame.

    bin / kitti : a single file IS the frame; timestamp/index ignored.
    mcap        : pick the frame nearest `timestamp` (Unix s) or `frame_index`.
                  For multi-topic MCAP the per-sensor raw clouds are cached in
                  _LAST_LOAD_CACHE for the calibration step that follows.
    """
    if WORKFLOW == "bin":
        return load_nuscenes_bin(BIN_PATH)
    elif WORKFLOW == "kitti":
        return load_kitti_bin(KITTI_PATH)
    elif WORKFLOW == "mcap":
        if MCAP_TOPIC is not None:
            cache = _build_mcap_index(MCAP_PATH)
            ts_list = cache["index"][MCAP_TOPIC]
            if timestamp is not None:
                idx = int(np.argmin(np.abs(ts_list - int(timestamp * 1e9))))
            else:
                idx = frame_index
            return load_mcap_frame(MCAP_PATH, topic=MCAP_TOPIC, frame_index=idx)
        return load_mcap_frame_fast(
            MCAP_PATH, topics=LIDAR_TOPICS,
            target_idx=frame_index,
            target_sec=timestamp,
        )
    else:
        raise ValueError(f"Unknown WORKFLOW '{WORKFLOW}'. Use 'bin', 'mcap', or 'kitti'.")


# ============================================================
# Build the list of frames to process THIS run.
# Each job is a (timestamp, frame_index) pair; one of them is None.
#   - mcap      -> one job per entry in MCAP_FRAME_TIMESTAMPS (or _INDICES)
#   - bin/kitti -> exactly one job (the single file)
# The bag is indexed only once (cached), so adding timestamps is cheap.
# ============================================================
if WORKFLOW == "mcap":
    if MCAP_FRAME_INDICES:
        FRAME_JOBS = [(None, int(i)) for i in MCAP_FRAME_INDICES]
    else:
        FRAME_JOBS = [(float(ts), None) for ts in MCAP_FRAME_TIMESTAMPS]
else:
    FRAME_JOBS = [(None, None)]   # bin / kitti: one file = one frame

assert FRAME_JOBS, (
    "No frames queued. For mcap, fill MCAP_FRAME_TIMESTAMPS "
    "(or MCAP_FRAME_INDICES) with at least one entry."
)
print(f"Workflow: {WORKFLOW}  |  frames queued this run: {len(FRAME_JOBS)}")

# ---- quick-look preview of the FIRST frame only ----
_ts0, _idx0 = FRAME_JOBS[0]
points = load_frame(timestamp=_ts0, frame_index=_idx0)
print(f"\nPreview frame: {len(points):,} points")
print(f"X range:   [{points[:,0].min():.1f}, {points[:,0].max():.1f}] m")
print(f"Y range:   [{points[:,1].min():.1f}, {points[:,1].max():.1f}] m")
print(f"Z range:   [{points[:,2].min():.1f}, {points[:,2].max():.1f}] m")

## 2. Quick look — BEV, side, front (first frame)

Sanity check that the **first** queued frame loaded with the right geometry.
For KITTI you should see a clean ring pattern from the HDL-64E and a near-flat
ground; for the bike MCAP you'll see the three sensors overlapping. The full
batch (every timestamp) is processed later in section 6.

In [ ]:
def bev_plot(points, title="BEV", lim=40, c=None, cmap="viridis",
             cbar_label=None, ax=None, s=1):
    """Top-down scatter of a point cloud. If `c` is None, colour by height."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))
    if c is None:
        c = points[:, 2]
    sc = ax.scatter(points[:, 0], points[:, 1], c=c, s=s, cmap=cmap)
    ax.set_xlabel("X forward [m]")
    ax.set_ylabel("Y left [m]")
    ax.set_title(title)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.grid(True, lw=0.3)
    plt.colorbar(sc, ax=ax, label=cbar_label or "Z [m]", shrink=0.8)
    return ax


def side_plot(points, title="Side view", lim=40, c=None, cmap="viridis",
              cbar_label=None, ax=None, s=1):
    """Side projection: X (forward) vs Z (height)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4))
    if c is None:
        c = points[:, 1]
    sc = ax.scatter(points[:, 0], points[:, 2], c=c, s=s, cmap=cmap)
    ax.set_xlabel("X forward [m]")
    ax.set_ylabel("Z height [m]")
    ax.set_title(title)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-5, 10)
    ax.set_aspect("equal")
    ax.grid(True, lw=0.3)
    plt.colorbar(sc, ax=ax, label=cbar_label or "Y [m]", shrink=0.8)
    return ax


def front_plot(points, title="Front view", lim=40, c=None, cmap="viridis",
               cbar_label=None, ax=None, s=1):
    """Front projection: Y (left-right) vs Z (height)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4))
    if c is None:
        c = points[:, 0]
    sc = ax.scatter(points[:, 1], points[:, 2], c=c, s=s, cmap=cmap)
    ax.set_xlabel("Y left [m]")
    ax.set_ylabel("Z height [m]")
    ax.set_title(title)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-5, 10)
    ax.set_aspect("equal")
    ax.grid(True, lw=0.3)
    plt.colorbar(sc, ax=ax, label=cbar_label or "X [m]", shrink=0.8)
    return ax


fig, axes = plt.subplots(3, 1, figsize=(12, 16))
bev_plot(  points, title="Bird's-eye view — coloured by height",      ax=axes[0])
side_plot( points, title="Side view — coloured by Y (left-right)",    ax=axes[1])
front_plot(points, title="Front view — coloured by X (forward)",      ax=axes[2])
plt.tight_layout(); plt.show()


## 3. Stage 1a — Patchwork++ ground segmentation

We use the authors' official C++ implementation via `pypatchworkpp`.

**One-paragraph recap.** Patchwork++ splits the ground around the sensor into
**concentric zones**, each zone into **angular wedges** (CZM). Inside each
wedge it runs **R-GPF**: sort by height, take the lowest slice as seeds, fit
a plane through them with PCA, reject wedges whose plane is too vertical or
whose seeds are too spread in z, label everything within a distance threshold
of the plane as ground. The `++` adds Adaptive Ground Likelihood Estimation
(A-GLE), Temporal Ground Revert, and Reflected-Noise Removal.

> **Analogy.** Each wedge is a small patch of parking lot. You stand in it,
> squint at the ~20 lowest points, lay a ruler across them, and mark every
> other point within an inch of the ruler as floor. Everything else is obstacle.

In [ ]:
# ---------------------------------------------------------------
# PCA helper -- used by the optional eigenvalue+flatness post-filter.
# ---------------------------------------------------------------
def _pca_plane(pts):
    """Fit a plane through `pts` with PCA.

    Returns
    -------
    n        : (3,) unit normal, oriented so n_z >= 0
    c        : (3,) centroid
    eigvals  : (3,) eigenvalues, sorted DESCENDING (lam1 >= lam2 >= lam3)
    """
    pts = pts.astype(np.float64, copy=False)
    c = pts.mean(axis=0)
    cov = np.cov((pts - c).T)
    w, V = np.linalg.eigh(cov)          # ascending
    eigvals = w[::-1]                   # descending
    n = V[:, 0]                         # smallest eigval -> normal
    if n[2] < 0:
        n = -n
    return n, c, eigvals


def build_patchworkpp(config):
    params = pypatchworkpp.Parameters()
    params.sensor_height = float(config["sensor_height"])
    params.min_range     = float(config["min_range"])
    params.max_range     = float(config["max_range"])
    params.verbose       = bool(config.get("verbose", False))
    return pypatchworkpp.patchworkpp(params)


PatchworkPP = build_patchworkpp(SENSOR_CONFIG)
print(f"Patchwork++ configured with sensor_height = {SENSOR_CONFIG['sensor_height']} m, "
      f"range [{SENSOR_CONFIG['min_range']}, {SENSOR_CONFIG['max_range']}] m")


def segment_ground(points):
    """Run Patchwork++. Returns (ground, nonground) as (M, 3) arrays."""
    if points.shape[1] == 3:
        pts4 = np.hstack([points, np.zeros((len(points), 1))]).astype(np.float32)
    else:
        pts4 = points[:, :4].astype(np.float32)
    PatchworkPP.estimateGround(pts4)
    ground    = np.asarray(PatchworkPP.getGround())[:, :3]
    nonground = np.asarray(PatchworkPP.getNonground())[:, :3]
    return ground, nonground


In [ ]:
# ============================================================
# Per-frame Patchwork++  (+ /tf_static calibration for MCAP)
# ------------------------------------------------------------
# Returns:
#   raw_cal   : the cloud BEFORE Patchwork++  (calibrated into base_link
#               for multi-topic MCAP, otherwise the raw input)
#   ground_pw : Patchwork++ ground points     (pre-GLE)
#   nonground : Patchwork++ obstacle points
# ============================================================
def segment_and_calibrate(raw_points, sensor_cache):
    """sensor_cache: {topic: (raw_xyz, frame_id)} for multi-topic MCAP,
    or None for single-source (bin / kitti / single-topic MCAP)."""
    if WORKFLOW == "mcap" and MCAP_TOPIC is None and sensor_cache:
        tree = _build_tf_tree(MCAP_PATH)
        have_tf = bool(tree)

        g_parts, ng_parts, raw_parts = [], [], []
        for topic, (raw_xyz, frame_id) in sensor_cache.items():
            g, ng = segment_ground(raw_xyz)
            r = raw_xyz
            if have_tf:
                try:
                    R, t = _resolve_to_base(tree, frame_id, BASE_FRAME)
                    g  = _apply_tf(g,  R, t)
                    ng = _apply_tf(ng, R, t)
                    r  = _apply_tf(r,  R, t)   # calibrate the raw cloud too
                    print(f"  {topic}: {len(g):,} ground, {len(ng):,} obstacle  "
                          f"({frame_id} -> {BASE_FRAME})")
                except RuntimeError as e:
                    print(f"  [warn] {topic}: {e} -- using raw frame")
            else:
                print(f"  {topic}: {len(g):,} ground, {len(ng):,} obstacle  (uncalibrated)")
            g_parts.append(g); ng_parts.append(ng); raw_parts.append(r)

        ground    = np.concatenate(g_parts,   axis=0)
        nonground = np.concatenate(ng_parts,  axis=0)
        raw_cal   = np.concatenate(raw_parts, axis=0)
    else:
        # nuScenes .bin, KITTI .bin, or single-topic MCAP
        ground, nonground = segment_ground(raw_points)
        raw_cal = raw_points

    return raw_cal, ground, nonground


# ============================================================
# Before/after comparison plot for ONE frame.
# Top row = BEV, bottom row = side view.
# Left column = raw input (before), right column = Patchwork++ (after).
# ============================================================
def plot_before_after(raw_cal, ground, nonground, title_prefix=""):
    fig, axes = plt.subplots(2, 2, figsize=(15, 13))

    # --- BEV before (raw, coloured by height) ---
    sc = axes[0, 0].scatter(raw_cal[:, 0], raw_cal[:, 1], c=raw_cal[:, 2],
                            s=1, cmap="viridis")
    axes[0, 0].set_title(f"{title_prefix}BEFORE - raw input (BEV)")
    plt.colorbar(sc, ax=axes[0, 0], label="Z [m]", shrink=0.8)

    # --- BEV after (ground vs obstacle) ---
    axes[0, 1].scatter(nonground[:, 0], nonground[:, 1], s=1, c="crimson",     label="Obstacle")
    axes[0, 1].scatter(ground[:, 0],    ground[:, 1],    s=1, c="forestgreen", label="Ground")
    axes[0, 1].set_title(f"{title_prefix}AFTER - Patchwork++ (BEV)")
    axes[0, 1].legend(markerscale=5)

    # --- Side before ---
    sc = axes[1, 0].scatter(raw_cal[:, 0], raw_cal[:, 2], c=raw_cal[:, 1],
                            s=1, cmap="viridis")
    axes[1, 0].set_title("BEFORE - raw input (Side)")
    plt.colorbar(sc, ax=axes[1, 0], label="Y [m]", shrink=0.8)

    # --- Side after ---
    axes[1, 1].scatter(nonground[:, 0], nonground[:, 2], s=1, c="crimson",     label="Obstacle")
    axes[1, 1].scatter(ground[:, 0],    ground[:, 2],    s=1, c="forestgreen", label="Ground")
    axes[1, 1].set_title("AFTER - Patchwork++ (Side)")
    axes[1, 1].legend(markerscale=5)

    for ax in (axes[0, 0], axes[0, 1]):
        ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]")
        ax.set_xlim(-40, 40); ax.set_ylim(-40, 40)
        ax.set_aspect("equal"); ax.grid(True, lw=0.3)
    for ax in (axes[1, 0], axes[1, 1]):
        ax.set_xlabel("X forward [m]"); ax.set_ylabel("Z height [m]")
        ax.set_xlim(-40, 40); ax.set_ylim(-5, 10)
        ax.set_aspect("equal"); ax.grid(True, lw=0.3)

    plt.tight_layout(); plt.show()

## 4. (Optional) GLE eigenvalue + flatness post-filter

A second, stricter pass over the Patchwork++ ground. Useful for noisy frames
where you can afford to throw away some valid ground. **Off by default**
because the very signal we want to measure downstream (curvature) is what
this filter is built to reject.

In [ ]:
# ============================================================
# Optional eigenvalue + flatness post-filter
# ------------------------------------------------------------
# Tile the ground with a Cartesian grid, fit a PCA plane in each
# cell, drop the whole cell if it is too "fluffy" (lambda3 share
# too large) or too tilted (|n_z| too small).
# ============================================================
def eigen_flatness_filter(ground_xyz,
                          cell_size=1.0,
                          min_pts_per_cell=12,
                          flatness_thresh=0.03,
                          uprightness_thresh=0.85):
    """Return (kept, dropped, stats_dict)."""
    if len(ground_xyz) == 0:
        empty = {"cx": np.array([]), "cy": np.array([]),
                 "sigma": np.array([]), "nz": np.array([]),
                 "kept": np.array([], dtype=bool)}
        return ground_xyz, ground_xyz, empty

    x = ground_xyz[:, 0]; y = ground_xyz[:, 1]
    ix = np.floor(x / cell_size).astype(np.int64)
    iy = np.floor(y / cell_size).astype(np.int64)

    # Encode (ix, iy) as a single key for groupby
    keys = ix.astype(np.int64) * (1 << 32) + iy.astype(np.int64) + (1 << 31)
    order = np.argsort(keys)
    keys_sorted = keys[order]
    pts_sorted  = ground_xyz[order]

    # Group boundaries
    starts = np.concatenate([[0], np.where(np.diff(keys_sorted) != 0)[0] + 1,
                             [len(keys_sorted)]])

    kept_mask  = np.zeros(len(ground_xyz), dtype=bool)
    cx_list, cy_list, sigma_list, nz_list, keep_list = [], [], [], [], []

    for a, b in zip(starts[:-1], starts[1:]):
        idx_cell = order[a:b]
        sub = pts_sorted[a:b]
        n_pts = len(sub)
        if n_pts < min_pts_per_cell:
            kept_mask[idx_cell] = True
            continue
        n, c, eigvals = _pca_plane(sub)
        s = float(eigvals[2] / (eigvals.sum() + 1e-12))
        nz = float(abs(n[2]))
        keep = (s <= flatness_thresh) and (nz >= uprightness_thresh)
        if keep:
            kept_mask[idx_cell] = True
        cx_list.append(c[0]); cy_list.append(c[1])
        sigma_list.append(s); nz_list.append(nz); keep_list.append(keep)

    stats = {
        "cx":   np.asarray(cx_list),
        "cy":   np.asarray(cy_list),
        "sigma":np.asarray(sigma_list),
        "nz":   np.asarray(nz_list),
        "kept": np.asarray(keep_list, dtype=bool),
    }
    return ground_xyz[kept_mask], ground_xyz[~kept_mask], stats


def plot_eigen_flatness_diagnostic(stats, cell_size, flatness_thresh, uprightness_thresh):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    colours = np.where(stats["kept"], "forestgreen", "crimson")
    axes[0].scatter(stats["cx"], stats["cy"], c=colours,
                    s=(cell_size * 9) ** 2, marker="s")
    axes[0].plot(0, 0, "o", color="dodgerblue", ms=10)
    axes[0].set_title("Cell kept (green) vs dropped (red)")

    sc1 = axes[1].scatter(stats["cx"], stats["cy"], c=stats["sigma"],
                          cmap="magma", s=(cell_size * 9) ** 2, marker="s",
                          vmin=0, vmax=max(0.05, 2 * flatness_thresh))
    axes[1].plot(0, 0, "o", color="dodgerblue", ms=10)
    axes[1].set_title(r"Flatness $\sigma=\lambda_3/\sum\lambda$"
                      + f"\n(threshold = {flatness_thresh})")
    plt.colorbar(sc1, ax=axes[1], shrink=0.7, label=r"$\sigma$")

    sc2 = axes[2].scatter(stats["cx"], stats["cy"], c=stats["nz"],
                          cmap="viridis", s=(cell_size * 9) ** 2, marker="s",
                          vmin=0.5, vmax=1.0)
    axes[2].plot(0, 0, "o", color="dodgerblue", ms=10)
    axes[2].set_title(r"Uprightness $|n_z|$"
                      + f"\n(threshold = {uprightness_thresh})")
    plt.colorbar(sc2, ax=axes[2], shrink=0.7, label="|n_z|")

    for ax in axes:
        ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]")
        ax.set_aspect("equal"); ax.grid(True, lw=0.3)
    plt.tight_layout(); plt.show()


# ============================================================
# Wrap the optional GLE pass so the batch loop can call it per frame.
# Returns the kept ground points (unchanged if the filter is off).
# ============================================================
def apply_gle(ground_pw, plot=False):
    if not APPLY_EIGEN_FLATNESS_FILTER:
        return ground_pw

    kept, dropped, stats = eigen_flatness_filter(
        ground_pw,
        cell_size=POST_CELL_SIZE,
        min_pts_per_cell=POST_MIN_PTS_PER_CELL,
        flatness_thresh=FLATNESS_THRESHOLD,
        uprightness_thresh=UPRIGHTNESS_THRESHOLD,
    )
    print(f"  [GLE] points kept {len(kept):,} / {len(ground_pw):,} "
          f"(dropped {len(dropped):,}; "
          f"cells dropped {int((~stats['kept']).sum())})")
    if plot:
        plot_eigen_flatness_diagnostic(stats, POST_CELL_SIZE,
                                       FLATNESS_THRESHOLD, UPRIGHTNESS_THRESHOLD)
    return kept

## 5. Export helpers — raw, ground-only, **and** full cloud

Three `.bin` files per frame, all nuScenes-format `(N, 5)` `float32` so they
round-trip cleanly:

| File | Contents | When | Intensity meaning | Use it for |
|---|---|---|---|---|
| `*_raw.bin`     | full scene | **before** PW++ | always `0.0` | before/after comparison |
| `*_pw.bin`      | ground only | **after** PW++ | always `0.0` | **Notebook 2 input** |
| `*_pw_full.bin` | ground + obstacle | **after** PW++ | `0.0` ground / `1.0` obstacle | Foxglove visualisation |

Filenames are workflow-aware so different sources never overwrite each other,
and the MCAP stem carries the sub-second part so two timestamps in the same
second stay distinct:

* KITTI:    `ground_points_pw_kitti_<basename>.bin`
* nuScenes: `ground_points_pw_nusc_<basename>.bin`
* MCAP:     `ground_points_pw_<date>_<time>_<unix>_<ms>.bin`

If `APPLY_EIGEN_FLATNESS_FILTER` is `True`, `_pw` becomes `_pw_gle` so you can
A/B the two filter strengths side by side. (`_raw` never gets the GLE suffix —
it is the pre-filter cloud.)

The cell below only *defines* the save/export helpers. The actual writing
happens once per timestamp in the batch loop in section 6.

In [ ]:
def save_as_nuscenes_bin(points_xyz, path, intensity=0.0, ring=0):
    """Save (N, 3) XYZ as nuScenes-format .bin -- (N, 5) float32 with
    intensity (scalar OR per-point array) and ring padding."""
    n = len(points_xyz)
    arr = np.zeros((n, 5), dtype=np.float32)
    arr[:, :3] = points_xyz.astype(np.float32)
    if np.isscalar(intensity):
        arr[:, 3] = intensity
    else:
        intensity = np.asarray(intensity, dtype=np.float32).reshape(-1)
        if len(intensity) != n:
            raise ValueError(f"intensity length {len(intensity)} != n_points {n}")
        arr[:, 3] = intensity
    arr[:, 4] = ring
    arr.tofile(path)
    print(f"  saved {n:,} pts -> {os.path.basename(path)}  ({arr.nbytes / 1024:.1f} KB)")


# ============================================================
# Build a workflow-aware filename stem (per frame)
# ============================================================
import datetime

def _filename_stem(timestamp=None, frame_index=None):
    if WORKFLOW == "kitti":
        base = os.path.splitext(os.path.basename(KITTI_PATH))[0]
        return f"kitti_{base}"
    if WORKFLOW == "bin":
        base = os.path.splitext(os.path.basename(BIN_PATH))[0]
        return f"nusc_{base}"
    # mcap: prefer the timestamp; resolve an index to a timestamp if needed
    if timestamp is None and frame_index is not None:
        cache = _build_mcap_index(MCAP_PATH)
        ref_topic = MCAP_TOPIC or LIDAR_TOPICS[0]
        timestamp = cache["index"][ref_topic][frame_index] / 1e9
    _ts = datetime.datetime.fromtimestamp(timestamp)
    rounded_minute = (_ts.minute // 5) * 5
    date_folder = _ts.strftime("%Y_%m_%d")
    time_folder = f"{_ts.strftime('%H')}_{rounded_minute:02d}_00"
    unix_t = int(timestamp)
    ms = int(round((timestamp - unix_t) * 1000))     # keep frames in same second distinct
    return f"{date_folder}_{time_folder}_{unix_t}_{ms:03d}"


_suffix = "_pw_gle" if APPLY_EIGEN_FLATNESS_FILTER else "_pw"
os.makedirs(GROUND_BIN_DIR, exist_ok=True)


# ============================================================
# Three exporters -- each returns the written path (or None if disabled).
# ============================================================
def export_raw(raw_cal, stem):
    """Cloud BEFORE Patchwork++ (calibrated into base_link for MCAP).
    intensity stays 0.0 -- this is the unfiltered reference cloud."""
    if not EXPORT_RAW_CLOUD:
        return None
    path = os.path.join(GROUND_BIN_DIR, f"{GROUND_BIN_BASE}_raw_{stem}.bin")
    save_as_nuscenes_bin(raw_cal, path)
    return path


def export_ground(ground, stem):
    """Ground-only cloud AFTER Patchwork++ (Notebook 2 input)."""
    if not EXPORT_GROUND:
        return None
    path = os.path.join(GROUND_BIN_DIR, f"{GROUND_BIN_BASE}{_suffix}_{stem}.bin")
    save_as_nuscenes_bin(ground, path)
    reloaded = np.fromfile(path, dtype=np.float32).reshape(-1, 5)
    assert len(reloaded) == len(ground), "Round-trip count mismatch!"
    return path


def export_full(ground, nonground, stem):
    """Full labelled cloud AFTER Patchwork++ (Foxglove inspection).
    intensity = 0.0 ground / 1.0 obstacle."""
    if not EXPORT_FULL_CLOUD:
        return None
    full_xyz = np.concatenate([ground, nonground], axis=0)
    full_lab = np.concatenate([
        np.zeros(len(ground),    dtype=np.float32),   # 0 = ground
        np.ones( len(nonground), dtype=np.float32),   # 1 = obstacle
    ])
    path = os.path.join(GROUND_BIN_DIR, f"{GROUND_BIN_BASE}{_suffix}_full_{stem}.bin")
    save_as_nuscenes_bin(full_xyz, path, intensity=full_lab)
    return path

## 6. Batch over every timestamp

This is the cell you actually run. For each entry in `FRAME_JOBS` it loads the
frame, runs Patchwork++ (+ calibration), applies the optional GLE filter,
shows a before/after plot, and writes the three `.bin` files. The expensive
bag index and `/tf_static` tree are built on the first frame and reused for
the rest, so adding more timestamps to `MCAP_FRAME_TIMESTAMPS` costs only the
windowed read + segmentation + save for each one.

In [ ]:
# ============================================================
# BATCH DRIVER -- process every frame in FRAME_JOBS in one run.
# The bag index and /tf_static tree are built once and reused, so the
# only per-frame cost is the (windowed) read + Patchwork++ + save.
# ============================================================
MANIFEST = []   # one dict per frame: {label, stem, raw, ground, full}

for k, (ts, idx) in enumerate(FRAME_JOBS, start=1):
    label = (f"t={ts:.3f}s" if ts is not None
             else (f"frame #{idx}" if idx is not None else WORKFLOW))
    print("=" * 64)
    print(f"[{k}/{len(FRAME_JOBS)}]  {label}")
    print("=" * 64)

    # 1) load the raw frame (windowed read -- fast)
    raw_points = load_frame(timestamp=ts, frame_index=idx)
    sensor_cache = (dict(_LAST_LOAD_CACHE)
                    if (WORKFLOW == "mcap" and MCAP_TOPIC is None) else None)

    # 2) Patchwork++ (+ calibration), then optional GLE
    raw_cal, ground_pw, nonground = segment_and_calibrate(raw_points, sensor_cache)
    ground = apply_gle(ground_pw, plot=PLOT_EACH_FRAME)

    tot = len(ground) + len(nonground)
    frac = (100.0 * len(ground) / tot) if tot else 0.0
    print(f"  raw {len(raw_cal):,} | ground {len(ground):,} | "
          f"obstacle {len(nonground):,} | ground fraction {frac:.1f}%")

    # 3) before/after plot
    if PLOT_EACH_FRAME:
        plot_before_after(raw_cal, ground, nonground, title_prefix=f"[{label}] ")

    # 4) export all requested files for this frame
    stem        = _filename_stem(timestamp=ts, frame_index=idx)
    raw_path    = export_raw(raw_cal,            stem)
    ground_path = export_ground(ground,          stem)
    full_path   = export_full(ground, nonground, stem)

    MANIFEST.append({"label": label, "stem": stem,
                     "raw": raw_path, "ground": ground_path, "full": full_path})

print("\nAll frames done.")

In [ ]:
# ============================================================
# Summary of everything written this run.
# Copy a "ground (after)" path below into Notebook 2's INPUT_BIN_PATH.
# ============================================================
print(f"Wrote files for {len(MANIFEST)} frame(s) into:\n  {GROUND_BIN_DIR}\n")
for m in MANIFEST:
    print(f"- {m['label']}")
    if m["raw"]:
        print(f"    raw    (before): {os.path.basename(m['raw'])}")
    if m["ground"]:
        print(f"    ground (after) : {os.path.basename(m['ground'])}   <- Notebook 2 input")
    if m["full"]:
        print(f"    full   (after) : {os.path.basename(m['full'])}   <- Foxglove (raw vs full = before/after)")
    print()

## What this notebook is for in the BEP report

This notebook isolates **filtration** as a single experimental variable, and
emits three artefacts per frame:

* `*_raw.bin` — the scene *before* Patchwork++ (the unfiltered control),
* `*_pw.bin` — the ground-only cloud Notebook 2 consumes for slope/curvature,
* `*_pw_full.bin` — the full labelled cloud for Foxglove inspection.

The `_raw` vs `_pw_full` pair is the **before/after** evidence: drop both into
Foxglove and you can show, frame by frame, exactly which points the filter
classified as ground and which it removed. Processing a batch of timestamps in
one run makes it cheap to report the filter's behaviour across many scenes
rather than a single cherry-picked frame.

Together with the v7 MCAP outputs and the collaborator's RANSAC notebook,
this gives the report a clean 3-source x 2-filter table:

| Source  | Patchwork++ | Patchwork++ + GLE |
|---|---|---|
| KITTI (flat baseline) | ... | ... |
| nuScenes              | ... | ... |
| SenseBike (Dutch hills) | ... | ... |

The KITTI row is the **flat negative control** — you expect kappa ~ 0 almost
everywhere. Large reported curvatures on KITTI are false positives to chase
down; small ones validate that the pipeline doesn't hallucinate signal where
there is none.